# KinoCompass – векторы e5 для поиска на сайте

Шаг 6д: векторы multilingual-e5-small для всех 23 349 фильмов каталога по **всем** тегам набора (сайт знает всё до 12.10.2023) – среднее векторов английского и русского текста «название. жанры. теги. описание», как лучший вариант проверки поиска (шаг 5в). На GPU – минуты, на процессоре – около часа.

Запускает `python export.py e5`; в конце скачается архив `export_e5.zip` (`e5_enru.npy` – векторы float16, `e5_ids.npy` – id фильмов) – его нужно распаковать в `model/export/`.

**Как запустить:** «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU**, затем «Среда выполнения» → «Выполнить все».

Время на T4 (оценка): загрузка набора – около 1 минуты, модель e5 – около минуты, векторы – 3–8 минут.

In [1]:
# каталог – где хранить модель e5 и посчитанные векторы внутри Colab
CACHE = "/content/cache"

In [2]:
import torch

assert torch.cuda.is_available(), "нет GPU: Среда выполнения -> Сменить среду выполнения -> T4 GPU"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "ГБ")

Tesla T4 14.6 ГБ


In [3]:
# из /content: при повторном запуске в том же сеансе текущая папка – внутри удаляемого клона
%cd /content
!rm -rf /content/KinoCompass && git clone -q --depth 1 https://github.com/AlexTrav/KinoCompass.git /content/KinoCompass
!git -C /content/KinoCompass log --oneline -1
# те же версии, что в образе Docker (torch в Colab уже есть)
!pip install -q requests==2.34.2 pandas==3.1.0 pyarrow==26.0.1 scipy==1.18.1 sentence-transformers==6.2.0

/content
efb44e1 (grafted, HEAD -> main, origin/main, origin/HEAD) Добавил выгрузку моделей для сайта: популярность, ALS, TF-IDF, каталог и модель запросов e5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.0 which is incompatible.


In [4]:
%cd /content/KinoCompass/model
!PYTHONPATH=/content/KinoCompass/data:/content/KinoCompass/research CACHE={CACHE} python -W ignore export.py e5

/content/KinoCompass/model
скачиваю https://files.grouplens.org/datasets/movielens/ml-32m.zip в память
  0 % (1 из 227 МБ)
  25 % (57 из 227 МБ)
  50 % (114 из 227 МБ)
  75 % (171 из 227 МБ)
  100 % (227 из 227 МБ)
  ratings: 32 000 204 строк
  tags: 2 000 072 строк
  movies: 87 585 строк
  links: 87 585 строк
e5: 23349 текстов
modules.json: 100% 387/387 [00:00<00:00, 1.68MB/s]
README.md: 100% 498k/498k [00:00<00:00, 207MB/s]
sentence_bert_config.json: 100% 57.0/57.0 [00:00<00:00, 218kB/s]
config.json: 100% 655/655 [00:00<00:00, 2.36MB/s]

model.safetensors: downloading bytes:  43% 202M/471M [00:01<00:01, 209MB/s, 16.2MB/s  ]
model.safetensors: downloading bytes:  67% 315M/471M [00:02<00:00, 226MB/s, 26.1MB/s  ]
model.safetensors: reconstructing file:  85% 402M/471M [00:06<00:01, 59.0MB/s, 12.7MB/s  ]
model.safetensors: downloading bytes: 100% 315M/315M [00:07<00:00, 43.7MB/s, 28.4MB/s  ]
model.safetensors: reconstructing file: 100% 471M/471M [00:07<00:00, 65.4MB/s, 36.7MB/s  ]
Loading

In [5]:
from google.colab import files

!cd /content/KinoCompass/model/export && zip -q /content/export_e5.zip e5_enru.npy e5_ids.npy && ls -la e5_enru.npy e5_ids.npy
files.download("/content/export_e5.zip")

-rw-r--r-- 1 root root 17932160 Nov 18 21:49 e5_enru.npy
-rw-r--r-- 1 root root    93524 Nov 18 21:49 e5_ids.npy


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>